In [32]:
# !pip install datasets

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("wangrongsheng/ag_news")

df = pd.DataFrame(dataset["train"])
df.head()


,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    120000 non-null  object
 1   label   120000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.8+ MB


In [34]:
df.describe()

,label
count,120000.000000
mean,1.500000
std,1.118039
min,0.000000
25%,0.750000
50%,1.500000
75%,2.250000
max,3.000000


In [35]:
df.isnull().sum()

,0
text,0
label,0


In [36]:
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [37]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

In [38]:
train_df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [39]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [40]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])

X_test = tokenizer.texts_to_sequences(test_df["text"])

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [42]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(train_df["label"], num_classes=4)

y_test = to_categorical(test_df["label"], num_classes=4)

In [43]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

embedding_dim = 128

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    ),

    SimpleRNN(128),

    Dense(4, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [44]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [45]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 62s 40ms/step - accuracy: 0.7936 - loss: 0.5873 - val_accuracy: 0.7933 - val_loss: 0.5597
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 62s 42ms/step - accuracy: 0.7008 - loss: 0.7237 - val_accuracy: 0.5054 - val_loss: 1.1748
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 79s 40ms/step - accuracy: 0.6642 - loss: 0.9009 - val_accuracy: 0.6660 - val_loss: 0.9075
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 61s 41ms/step - accuracy: 0.7038 - loss: 0.8224 - val_accuracy: 0.6796 - val_loss: 0.8515
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 40ms/step - accuracy: 0.6775 - loss: 0.8626 - val_accuracy: 0.6578 - val_loss: 0.9563


In [46]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6621 - loss: 0.9349
Test Loss: 0.9348908066749573
Test Accuracy: 0.6621052622795105


In [47]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, 50, 128)          │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (64, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, 4)                │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,940,238 (15.03 MB)

 Trainable params: 1,313,412 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,826 (10.02 MB)